# Notebook 2 · Agir avec des outils

*Formation « Comprendre et développer des agents IA spécialisés », module 2.*

Chaque jour, les clients de **Djassa** posent des questions qui ne se règlent pas avec du langage : *« L'iPhone 16 rose est dispo ? »*, *« Vous avez les Air Force 1 en 42 ? »*, *« Je cherche un cadeau à moins de 40 000 FCFA »*, *« Je le reçois quand à Cocody ? Je peux le renvoyer ? »* Pour y répondre juste, Awa doit connaître l'état réel du stock, les prix du jour et les règles de livraison. Or le notebook 1 a montré qu'un LLM ne sait rien de la boutique. Ce notebook lui donne les moyens de **consulter** le catalogue au lieu d'inventer.

<p align="center"><img src="assets/diagrams/nb2_mise_en_contexte.png" alt="Quatre questions de clients de Djassa : Awa, conseillère de vente IA, y répond en consultant le vrai catalogue (stock, prix, avis, livraison, retours) au lieu d'inventer" width="1000"></p>

À gauche, quatre questions typiques de clients ; au centre, Awa ; à droite, les données de la boutique qu'elle interroge (stock, prix, avis, livraison, retours). Les flèches indiquent le sens du travail : la question arrive, Awa choisit où chercher, puis formule une réponse **à partir de ce qu'elle a trouvé**. C'est ce passage de l'assistant qui parle à l'**agent** qui agit que construit ce notebook.

Quatre notions se succèdent : les **outils** (des fonctions que le modèle peut demander d'exécuter), le **tool calling** (le protocole de cette demande), la **boucle ReAct** (la répétition qui permet d'enchaîner plusieurs outils) et la **validation des arguments** (le contrôle que ton code exerce avant d'exécuter quoi que ce soit). Chacune est d'abord écrite **en Python pur**, puis reprise **avec LangChain ou LangGraph**.

<p align="center"><img src="assets/diagrams/nb2_parcours_agent.png" alt="De l'assistant à l'agent : outils (étape 3), tool calling (étape 4), boucle ReAct (étape 5), validation (étape 6), puis agent" width="1000"></p>

**Un périmètre volontairement limité.** Ici, Awa **consulte** : stock, prix, livraison, retours. Réserver un article, rembourser ou modifier une commande sont des actions à conséquences, qui demandent des autorisations, des contrôles métier et souvent une validation humaine. Elles font l'objet du notebook 4.

<p align="center"><img src="assets/diagrams/nb2_perimetre_awa.png" alt="Dans ce notebook, Awa consulte stock, prix, livraison et retours ; réserver, rembourser et modifier une commande sont des actions à risque traitées au notebook 4" width="900"></p>

---
## Étape 0 · Choisir le modèle qui anime Awa

Awa a besoin d'un LLM capable de **tool calling**, c'est-à-dire de demander l'exécution d'une fonction plutôt que de répondre par du texte. Deux options sont prévues :

- **`"openai"`** (par défaut) : un modèle hébergé par OpenAI, appelé via Internet. Il faut une clé `OPENAI_API_KEY` dans le fichier `.env`. C'est l'option la plus fiable pour des agents.
- **`"ollama"`** : un modèle installé sur ton ordinateur avec [Ollama](https://ollama.com). C'est gratuit et sans clé, mais les petits modèles locaux enchaînent moins bien les outils, ce qu'on observera à l'étape 7.

Seule la première ligne de la cellule suivante est à modifier.

In [1]:
FOURNISSEUR = "openai"          # "openai" (clé dans .env) ou "ollama" (local, gratuit)

import json, os
from datetime import date, timedelta
from dotenv import load_dotenv
from openai import OpenAI

AUJOURDHUI = date(2026, 10, 27)  # date fixe du scénario : mêmes résultats pour tout le monde

load_dotenv()
if FOURNISSEUR == "ollama":     # Ollama parle le même « langage » que l'API OpenAI
    MODELE, BASE_URL, CLE = "qwen2.5:7b", "http://localhost:11434/v1", "ollama"
else:
    MODELE, BASE_URL, CLE = "gpt-5-mini", None, os.getenv("OPENAI_API_KEY")
REGLAGES = {"temperature": 0} if FOURNISSEUR == "ollama" else {}   # réponses plus stables en local

client = OpenAI(base_url=BASE_URL, api_key=CLE)
print(f"Prêt : {FOURNISSEUR} / {MODELE}")

Prêt : openai / gpt-5-mini


Avant d'écrire l'agent, une petite fonction d'affichage rend ses traces lisibles. Chaque ligne porte une étiquette : **Note** (un texte intermédiaire du LLM, rare), **Appelle** (l'agent demande un outil), **Reçoit** (il reçoit le résultat de l'outil) et **Répond** (il s'adresse au client). Cette trace sera notre principal instrument d'observation : elle permet de voir ce que l'agent décide, ce qu'on appelle l'**observabilité**.

In [2]:
ICONES = {"pensee": "Note", "outil": "Appelle", "resultat": "Reçoit", "reponse": "Répond"}

def afficher(etape, texte):
    """Affiche une ligne de trace, par exemple : Appelle │ verifier_stock(...)"""
    if etape == "reponse":
        print()
    print(f"{ICONES[etape]:<12} │ {texte}")

afficher("outil", "exemple d'affichage")

Appelle      │ exemple d'affichage


---
## Étape 1 · Le catalogue Djassa

Un agent n'est utile que s'il a accès à **tes** données. On commence donc par les fabriquer. Dans une vraie boutique, elles viendraient d'une base de données ou d'une plateforme e-commerce comme Shopify ou PrestaShop ; ici, de simples **dictionnaires Python** jouent ce rôle, pour que tout reste visible dans le notebook.

<p align="center"><img src="assets/diagrams/nb2_catalogue_djassa.png" alt="Le catalogue Djassa : les informations de la boutique, 9 produits répartis en 3 rayons, et le détail d'un produit avec ses prix, sa fiche, son stock et ses avis" width="1000"></p>

Trois niveaux d'information s'emboîtent. En haut, les **informations de la boutique** (nom, slogan, seuil de livraison gratuite). Au milieu, **9 produits en 3 rayons** (high-tech, mode, quotidien). À droite, le détail d'**un produit** : son prix (et son ancien prix s'il est en promotion), sa fiche, son stock par couleur ou par couleur et taille, et les avis des clients. Les outils de l'étape 3 seront autant de façons d'interroger ces trois niveaux.

On commence par les informations générales de la boutique :

In [3]:
BOUTIQUE = {
    "nom": "Djassa",
    "slogan": "Les bons plans du quotidien, livrés partout en Côte d'Ivoire",
    "livraison_gratuite_des": 50000,         # en FCFA, pour la livraison standard
    "service_client": "du lundi au samedi, de 8 h à 18 h, au +225 07 00 00 00 00 (appel ou WhatsApp)",
}
# Règles de livraison de la boutique (fictives, pour la démonstration) : (jours ouvrés, frais en FCFA)
ZONES = {
    "Abidjan":   {"villes": ["Abobo", "Adjamé", "Attécoubé", "Cocody", "Koumassi", "Marcory", "Plateau", "Port-Bouët",
                             "Treichville", "Yopougon"],
                  "standard": (2, 1500), "express": (1, 2500)},
    "Intérieur": {"villes": ["Bouaké", "Yamoussoukro", "San Pedro"],
                  "standard": (4, 3000), "express": (2, 5000)},
}
POLITIQUE_RETOUR = ("Retour sous 14 jours, article non utilisé et dans son emballage d'origine (gratuit en point relais Djassa). "
                    "Remboursement sous 5 jours ouvrés après réception du colis, sur le moyen de paiement utilisé.")
MOYENS_PAIEMENT = ["Orange Money", "MTN MoMo", "Wave", "carte bancaire", "paiement à la livraison (Abidjan uniquement)",
                   "paiement en 3 fois dès 150 000 FCFA"]

def fcfa(montant):
    """45000 -> '45 000 FCFA' : montants entiers, milliers séparés par une espace."""
    return f"{montant:,}".replace(",", " ") + " FCFA"

CATALOGUE = {}                               # rempli dans les 3 cellules suivantes

Viennent ensuite les produits. Tous partagent la même structure : le prix (avec `prix_barre` pour l'ancien prix en cas de promotion), la fiche (`description`, `caracteristiques`), le `stock` et les avis (`note`, `nombre_avis`, `avis`). Le stock est donné par couleur, et **par couleur puis par taille** pour les vêtements et les chaussures, car une paire de baskets blanches existe en 42 sans exister en 44.

Ces trois cellules sont des cellules de **données** : inutile de les lire en détail, exécute-les simplement.

In [4]:
# Rayon HIGH-TECH
CATALOGUE["iphone 16"] = {
    "marque": "Apple", "categorie": "Téléphones", "prix": 690000, "prix_barre": 730000,
    "description": "Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,1 pouces", "photo": "48 MP", "garantie": "2 ans"},
    "stock": {"noir": 8, "blanc": 5, "rose": 0, "turquoise": 3},
    "note": 4.7, "nombre_avis": 328,
    "avis": [{"auteur": "Fatou", "note": 5, "texte": "Très bon téléphone, rapide et excellent en photo."},
             {"auteur": "Kouamé", "note": 4, "texte": "Très agréable à utiliser, mais le prix reste élevé."}],
}
CATALOGUE["samsung galaxy a36"] = {
    "marque": "Samsung", "categorie": "Téléphones", "prix": 245000, "prix_barre": 265000,
    "description": "Écran AMOLED 6,7 pouces, triple appareil photo et batterie qui tient la journée. Un très bon rapport qualité-prix.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,7 pouces", "photo": "50 MP", "garantie": "2 ans"},
    "stock": {"noir": 6, "bleu": 4, "argent": 2},
    "note": 4.5, "nombre_avis": 214,
    "avis": [{"auteur": "Salimata", "note": 5, "texte": "Écran magnifique et téléphone très fluide."},
             {"auteur": "Ibrahim", "note": 4, "texte": "Très complet, notamment pour les photos."}],
}
CATALOGUE["écouteurs airpods 4"] = {
    "marque": "Apple", "categorie": "Audio", "prix": 125000,
    "description": "Écouteurs sans fil avec audio spatial et boîtier de recharge USB-C.",
    "caracteristiques": {"autonomie": "5 h (30 h avec le boîtier)", "connexion": "Bluetooth", "micro": "oui"},
    "stock": {"blanc": 12},
    "note": 4.5, "nombre_avis": 192,
    "avis": [{"auteur": "Adjoua", "note": 5, "texte": "Connexion simple et son agréable."},
             {"auteur": "Cheick", "note": 4, "texte": "Très pratiques dans les transports."}],
}
CATALOGUE["montre connectée xiaomi"] = {
    "marque": "Xiaomi", "categorie": "Montres", "prix": 32000, "prix_barre": 38000,
    "description": "Suivi d'activité, fréquence cardiaque, sommeil et notifications. Compatible Android et iPhone.",
    "caracteristiques": {"écran": "AMOLED", "autonomie": "14 jours", "étanchéité": "5 ATM"},
    "stock": {"noir": 9, "argent": 4},
    "note": 4.2, "nombre_avis": 231,
    "avis": [{"auteur": "Eric", "note": 5, "texte": "Bonne autonomie et suivi sportif pratique."},
             {"auteur": "Nadège", "note": 3, "texte": "Bon rapport qualité-prix, application perfectible."}],
}

In [5]:
# Rayon MODE : le stock est donné par couleur PUIS par taille
CATALOGUE["baskets nike air force 1"] = {
    "marque": "Nike", "categorie": "Chaussures", "prix": 75000, "prix_barre": 82000,
    "description": "Baskets emblématiques en cuir, semelle épaisse, design intemporel. Unisexe.",
    "caracteristiques": {"matière": "cuir", "style": "lifestyle", "entretien": "chiffon humide"},
    "stock": {"blanc": {"39": 2, "40": 4, "41": 5, "42": 8, "43": 3, "44": 2, "45": 0},
              "noir":  {"39": 1, "40": 3, "41": 2, "42": 0, "43": 5, "44": 2, "45": 1}},
    "note": 4.6, "nombre_avis": 512,
    "avis": [{"auteur": "Franck", "note": 5, "texte": "Très belles baskets, faciles à porter."},
             {"auteur": "Désirée", "note": 4, "texte": "Modèle classique, mais un peu lourd."}],
}
CATALOGUE["hoodie adidas essentials"] = {
    "marque": "Adidas", "categorie": "Vêtements", "prix": 38000, "prix_barre": 45000,
    "description": "Sweat à capuche confortable avec poche kangourou et coupe décontractée.",
    "caracteristiques": {"matière": "coton et polyester", "coupe": "décontractée", "lavage": "30 °C"},
    "stock": {"noir": {"S": 3, "M": 5, "L": 4, "XL": 2}, "gris": {"S": 2, "M": 4, "L": 3, "XL": 1},
              "beige": {"S": 1, "M": 0, "L": 2, "XL": 2}},
    "note": 4.4, "nombre_avis": 146,
    "avis": [{"auteur": "Jean-Marc", "note": 5, "texte": "Très confortable et agréable à porter."},
             {"auteur": "Clarisse", "note": 3, "texte": "Belle qualité, mais taille un peu grand."}],
}
CATALOGUE["jean levis 501"] = {
    "marque": "Levi's", "categorie": "Vêtements", "prix": 55000, "prix_barre": 62000,
    "description": "Jean droit iconique à boutons, en denim résistant. Un classique du quotidien.",
    "caracteristiques": {"coupe": "droite", "matière": "denim", "lavage": "basse température"},
    "stock": {"bleu": {"36": 2, "38": 4, "40": 5, "42": 3, "44": 1}, "noir": {"36": 1, "38": 3, "40": 0, "42": 2, "44": 2}},
    "note": 4.3, "nombre_avis": 278,
    "avis": [{"auteur": "Stéphane", "note": 5, "texte": "Une coupe classique qui dure dans le temps."},
             {"auteur": "Rokia", "note": 4, "texte": "Très joli, mais vérifiez bien la taille."}],
}

In [6]:
# Rayon QUOTIDIEN
CATALOGUE["sac à dos eastpak"] = {
    "marque": "Eastpak", "categorie": "Sacs", "prix": 35000,
    "description": "Sac à dos compact de 24 litres, poche avant zippée et dos rembourré. Pour les cours ou le travail.",
    "caracteristiques": {"capacité": "24 L", "compartiment ordinateur": "non", "garantie": "30 ans"},
    "stock": {"noir": 10, "bleu": 4, "vert": 2},
    "note": 4.7, "nombre_avis": 346,
    "avis": [{"auteur": "Brice", "note": 5, "texte": "Solide, léger et parfait pour les cours."},
             {"auteur": "Estelle", "note": 4, "texte": "Très pratique, mais pas de compartiment ordinateur."}],
}
CATALOGUE["gobelet stanley quencher"] = {
    "marque": "Stanley", "categorie": "Maison", "prix": 28000,
    "description": "Gobelet isotherme de 1,18 L avec poignée, paille et couvercle. Garde le froid jusqu'à 11 h.",
    "caracteristiques": {"capacité": "1,18 L", "matière": "acier inoxydable", "lave-vaisselle": "oui"},
    "stock": {"rose": 3, "crème": 5, "noir": 7},
    "note": 4.5, "nombre_avis": 183,
    "avis": [{"auteur": "Sandrine", "note": 5, "texte": "Grande capacité et poignée très pratique."},
             {"auteur": "Ismaël", "note": 4, "texte": "Joli produit, mais assez encombrant."}],
}

Voici la **vitrine** de Djassa. La mention PROMO signale les produits en promotion.

In [7]:
print(f"{BOUTIQUE['nom']} : {BOUTIQUE['slogan']}\n")
for nom, p in CATALOGUE.items():
    promo = "PROMO" if p.get("prix_barre") else "     "
    print(f"{promo} {p['categorie']:<10} │ {nom:<25} │ {fcfa(p['prix']):>12} │ note {p['note']} ({p['nombre_avis']} avis)")

Djassa : Les bons plans du quotidien, livrés partout en Côte d'Ivoire

PROMO Téléphones │ iphone 16                 │ 690 000 FCFA │ note 4.7 (328 avis)
PROMO Téléphones │ samsung galaxy a36        │ 245 000 FCFA │ note 4.5 (214 avis)
      Audio      │ écouteurs airpods 4       │ 125 000 FCFA │ note 4.5 (192 avis)
PROMO Montres    │ montre connectée xiaomi   │  32 000 FCFA │ note 4.2 (231 avis)
PROMO Chaussures │ baskets nike air force 1  │  75 000 FCFA │ note 4.6 (512 avis)
PROMO Vêtements  │ hoodie adidas essentials  │  38 000 FCFA │ note 4.4 (146 avis)
PROMO Vêtements  │ jean levis 501            │  55 000 FCFA │ note 4.3 (278 avis)
      Sacs       │ sac à dos eastpak         │  35 000 FCFA │ note 4.7 (346 avis)
      Maison     │ gobelet stanley quencher  │  28 000 FCFA │ note 4.5 (183 avis)


**À retenir**

Le catalogue est la **source de vérité** d'Awa : stock, prix et avis viennent de là, pas du modèle. Tout l'enjeu des étapes suivantes est de lui en ouvrir l'accès de façon contrôlée.

---
## Étape 2 · Pourquoi un LLM seul ne suffit pas

Avant de construire quoi que ce soit, mesurons le problème. On pose une question de client au LLM **sans lui donner accès au catalogue** et on regarde ce qu'il fait de son ignorance.

In [8]:
question = "Bonjour, chez Djassa, l'iPhone 16 rose est en stock ? Il coûte combien ?"

reponse = client.chat.completions.create(
    model=MODELE,
    messages=[{"role": "user", "content": question}],
    **REGLAGES,
)
afficher("reponse", reponse.choices[0].message.content)


Répond       │ Bonjour — je n’ai pas accès aux stocks et prix en temps réel, donc je ne peux pas vérifier directement si l’iPhone 16 rose est disponible chez Djassa ni son tarif actuel.

Pour que je vous aide au mieux, dites-moi :
- quel magasin Djassa (ville ou adresse) ?  
- quel modèle exactement (iPhone 16, 16 Plus, 16 Pro/Pro Max) et quelle capacité (ex. 128/256/512 Go) ?

Sinon, voici comment vérifier rapidement vous-même et un modèle de message à envoyer à la boutique :

1) Vérification rapide
- Allez sur le site web ou l’appli de Djassa, faites une recherche « iPhone 16 rose ».
- Appelez le magasin (numéro sur leur site) pour confirmation immédiate.
- Envoyez un e-mail ou un message via leur chat/WhatsApp si disponible.

2) Modèle de message / SMS à envoyer
« Bonjour, avez-vous en stock l’iPhone 16 rose (modèle [ex : 128 Go]) ? Quel est le prix TTC et la disponibilité en magasin ? Merci. »

3) Script téléphone (très court)
« Bonjour, je voudrais savoir si vous avez l’iPhone 16

**Ce qui s'est passé.** Le LLM ne peut pas savoir : il n'a jamais vu le stock de Djassa, et il ne peut pas deviner que l'iPhone rose est en rupture. Il a deux issues possibles, toutes deux mauvaises pour une boutique : **inventer** un prix ou une disponibilité (le client commande sur une promesse fausse), ou **avouer son ignorance** (le client part chez un concurrent). Dans les deux cas, le modèle est à court de moyens, pas de langage.

**L'idée** consiste à lui donner des **outils**, c'est-à-dire des fonctions Python qu'il peut **demander** d'exécuter. L'image est celle d'un vendeur qui, interrogé sur un article, va vérifier en réserve avant de répondre.

<p align="center"><img src="assets/diagrams/nb2_llm_seul_vs_agent.png" alt="Un LLM seul invente sa réponse, alors qu'un agent consulte le catalogue grâce à ses outils" width="900"></p>

À gauche, le LLM seul répond directement : sa réponse est plausible mais n'a aucun lien avec le stock réel. À droite, l'agent intercale une étape : face à la même question, il **interroge le catalogue** au moyen d'un outil, reçoit un fait (« rupture de stock, couleurs disponibles : noir, blanc, turquoise »), et construit sa réponse sur ce fait. Le langage reste l'affaire du modèle ; la vérité vient de l'outil.

**À retenir**

Un LLM connaît le langage, pas ta boutique. Pour qu'il réponde juste, il faut lui donner un moyen de consulter les données.

---
## Étape 3 · Créer des outils

Un **outil** est une fonction Python ordinaire, avec une règle importante : elle **renvoie du texte avec `return`** (et non `print`), car c'est ce texte que le LLM lira ensuite pour formuler sa réponse.

On crée cinq outils, chacun répondant à un type de question de client :

| Outil | Question du client |
|---|---|
| `rechercher_produits` | « Vous avez quoi comme téléphones ? », « Un cadeau à moins de 40 000 FCFA ? » |
| `fiche_produit` | « Il vaut le coup ? Que disent les clients ? » |
| `verifier_stock` | « Vous l'avez en noir, en taille 42 ? » |
| `delai_livraison` | « Je le reçois quand ? » |
| `infos_pratiques` | « Je peux le renvoyer ? Je peux payer avec Orange Money ? » |

Ce découpage n'est pas arbitraire : un outil doit avoir **une responsabilité claire**, pour que le modèle sache lequel appeler à partir de la question. Un outil fourre-tout, qui fait dix choses, serait difficile à choisir et à décrire.

Une première difficulté pratique : le LLM n'écrit pas toujours les noms comme le catalogue. Il dira « Air Force » pour « baskets nike air force 1 », ou « bleue » pour « bleu ». Deux petites fonctions rendent la recherche tolérante à ces approximations (casse, accents, accords).

In [9]:
import unicodedata

def normaliser(texte):
    """'  Écharpe ' -> 'echarpe' : minuscules, sans accents ni apostrophes."""
    texte = unicodedata.normalize("NFD", texte.lower()).encode("ascii", "ignore").decode()
    return texte.replace("'", " ").split()

def trouver(recherche, choix, strict=False):
    """Renvoie l'élément de `choix` qui partage le plus de mots avec `recherche`, ou None.
    strict=True : tous les mots demandés doivent correspondre (on ne devine pas)."""
    mots = [m for m in normaliser(recherche) if len(m) > 2]
    def score(c):        # « bleue » correspond à « bleu », « telephones » à « telephone »...
        return sum(any(m.startswith(k) or k.startswith(m) for k in normaliser(c) if len(k) > 2) for m in mots)
    meilleur = max(choix, key=score, default=None)
    if meilleur is None or score(meilleur) == 0 or (strict and score(meilleur) < len(mots)):
        return None
    return meilleur

print(trouver("Air Force", CATALOGUE), "|", trouver("Levi's", CATALOGUE), "|", trouver("bleue", ["noir", "bleu"]))

baskets nike air force 1 | jean levis 501 | bleu


**Une tolérance a un prix.** Si un client demande des **Nike Air Max**, un modèle que Djassa ne vend pas, une recherche trop souple risque de répondre avec un autre produit.

In [10]:
print("Tolérant :", trouver("Nike Air Max", CATALOGUE))               # choisit un autre produit
print("Strict   :", trouver("Nike Air Max", CATALOGUE, strict=True))  # ne devine pas

Tolérant : baskets nike air force 1
Strict   : None


Le mode tolérant répond « Air Force 1 » : Awa risquerait d'annoncer un stock pour le **mauvais produit**, une erreur plus grave qu'un « je ne trouve pas ». Pour les produits, les outils utilisent donc la recherche **stricte** (`strict=True`) : en cas de doute, l'outil le dit, et Awa **demande une précision** au lieu de choisir au hasard. On garde la tolérance pour les attributs peu risqués, comme les couleurs.

### Les outils « catalogue »

In [11]:
INCONNU = f"Produit introuvable ou ambigu : demande au client de préciser. Produits vendus : {', '.join(CATALOGUE)}."

def rechercher_produits(recherche: str = "", budget_max: int = 0) -> str:
    """Cherche des produits par mot-clé, marque ou catégorie (ex. téléphone, chaussures, promo), avec un budget maximum en FCFA (0 = sans limite)."""
    dans_budget = {n: p for n, p in CATALOGUE.items() if not budget_max or p["prix"] <= budget_max}
    trouves = {n: p for n, p in dans_budget.items()
               if trouver(recherche, [f"{n} {p['marque']} {p['categorie']} {'promo' if p.get('prix_barre') else ''}"])}
    entete = "" if trouves or not recherche else f"Rien ne correspond à « {recherche} ». Produits dans le budget :\n"
    lignes = []
    for nom, p in sorted((trouves or dans_budget).items(), key=lambda x: -x[1]["note"]):   # mieux notés d'abord
        promo = f" au lieu de {fcfa(p['prix_barre'])}" if p.get("prix_barre") else ""
        lignes.append(f"- {nom} ({p['marque']}) : {fcfa(p['prix'])}{promo}, note {p['note']}/5 ({p['nombre_avis']} avis)")
    return entete + ("\n".join(lignes) or "Aucun produit dans ce budget.")

print(rechercher_produits("téléphone"))
print("\n")
print(rechercher_produits("le mieux noté", budget_max=40000))

- iphone 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA, note 4.7/5 (328 avis)
- samsung galaxy a36 (Samsung) : 245 000 FCFA au lieu de 265 000 FCFA, note 4.5/5 (214 avis)


Rien ne correspond à « le mieux noté ». Produits dans le budget :
- sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)


In [12]:
def fiche_produit(produit: str) -> str:
    """Donne la fiche complète d'un produit : prix, description, caractéristiques, couleurs en stock, note et avis clients."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    p = CATALOGUE[nom]
    prix = fcfa(p['prix']) + (f" au lieu de {fcfa(p['prix_barre'])}" if p.get("prix_barre") else "")
    en_stock = [c for c, q in p["stock"].items() if (sum(q.values()) if isinstance(q, dict) else q) > 0]
    lignes = [f"{nom.upper()} ({p['marque']}) : {prix}", p["description"],
              "Caractéristiques : " + ", ".join(f"{k} : {v}" for k, v in p["caracteristiques"].items()),
              f"Couleurs en stock : {', '.join(en_stock) or 'aucune'}. Note : {p['note']}/5 ({p['nombre_avis']} avis)."]
    lignes += [f"  « {a['texte']} » ({a['auteur']}, {a['note']}/5)" for a in p["avis"]]
    return "\n".join(lignes)

print(fiche_produit("Samsung"))

SAMSUNG GALAXY A36 (Samsung) : 245 000 FCFA au lieu de 265 000 FCFA
Écran AMOLED 6,7 pouces, triple appareil photo et batterie qui tient la journée. Un très bon rapport qualité-prix.
Caractéristiques : stockage : 128 Go, écran : 6,7 pouces, photo : 50 MP, garantie : 2 ans
Couleurs en stock : noir, bleu, argent. Note : 4.5/5 (214 avis).
  « Écran magnifique et téléphone très fluide. » (Salimata, 5/5)
  « Très complet, notamment pour les photos. » (Ibrahim, 4/5)


`verifier_stock` est un peu plus longue, car elle traite **deux structures de stock** : les produits à simple couleur (un iPhone noir) et ceux à couleur **et** taille (des baskets blanches en 42). Elle renvoie dans chaque cas un message exploitable, y compris en cas d'échec (« taille indisponible, tailles en stock : S, L, XL »), pour qu'Awa puisse rebondir par une alternative.

In [13]:
def verifier_stock(produit: str, couleur: str, taille: str = "") -> str:
    """Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    stock = CATALOGUE[nom]["stock"]
    teinte = trouver(couleur, stock)
    if teinte is None:
        return f"{nom} n'existe pas en {couleur}. Couleurs proposées : {', '.join(stock)}."
    if isinstance(stock[teinte], dict):                         # cas 2 : article avec tailles
        quantite = stock[teinte].get(taille.upper(), 0)
        dispo = [t for t, q in stock[teinte].items() if q > 0]
        if quantite == 0:
            return f"{nom} {teinte} : taille '{taille}' indisponible. Tailles en stock : {', '.join(dispo) or 'aucune'}."
        return f"{nom} {teinte} taille {taille.upper()} : {quantite} en stock."
    if stock[teinte] > 0:                                       # cas 1 : couleur seule
        return f"{nom} {teinte} : {stock[teinte]} en stock."
    autres = [c for c, q in stock.items() if q > 0]
    return f"{nom} {teinte} : rupture de stock. Couleurs disponibles : {', '.join(autres) or 'aucune'}."

print(verifier_stock("iPhone 16", "rose"))
print(verifier_stock("Air Force", "blanches", "42"))
print(verifier_stock("hoodie", "beige", "M"))

iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
baskets nike air force 1 blanc taille 42 : 8 en stock.
hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.


### Les outils « service client »

Les deux derniers outils ne lisent pas le stock mais les **règles de la boutique** : délais de livraison, conditions de retour, moyens de paiement. Le calcul des dates tient compte des jours ouvrés, ce qui évite de laisser le modèle faire de l'arithmétique sur des dates.

In [14]:
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

def dans_jours_ouvres(n):
    """Date dans n jours ouvrés (on saute samedi et dimanche), ex. 'jeudi 29/10'."""
    jour = AUJOURDHUI
    while n > 0:
        jour += timedelta(days=1)
        if jour.weekday() < 5:
            n -= 1
    return f"{JOURS[jour.weekday()]} {jour:%d/%m}"

def cle(texte):
    """'  Port-Bouët ' -> 'port bouet' : pour comparer sans se soucier des accents ni de la casse."""
    return " ".join(normaliser(texte.replace("-", " ")))

def delai_livraison(ville: str) -> str:
    """Estime les dates de livraison standard et express à domicile, à partir de la commune d'Abidjan du client (ex. Cocody) ou de sa ville (Bouaké, Yamoussoukro, San Pedro)."""
    zone = "Abidjan" if cle(ville) in [cle(v) for v in ZONES["Abidjan"]["villes"]] else "Intérieur"
    (jours_std, frais_std), (jours_exp, frais_exp) = ZONES[zone]["standard"], ZONES[zone]["express"]
    return (f"Standard : {dans_jours_ouvres(jours_std)} (offerte dès {fcfa(BOUTIQUE['livraison_gratuite_des'])}, sinon {fcfa(frais_std)}). "
            f"Express : {dans_jours_ouvres(jours_exp)} ({fcfa(frais_exp)}).")

In [15]:
def infos_pratiques() -> str:
    """Donne les conditions de retour et de remboursement, les moyens de paiement et les horaires du service client."""
    return (f"Retours : {POLITIQUE_RETOUR} Paiement : {', '.join(MOYENS_PAIEMENT)}. "
            f"Service client : {BOUTIQUE['service_client']}.")

print(delai_livraison("Cocody"))
print(infos_pratiques())

Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Retours : Retour sous 14 jours, article non utilisé et dans son emballage d'origine (gratuit en point relais Djassa). Remboursement sous 5 jours ouvrés après réception du colis, sur le moyen de paiement utilisé. Paiement : Orange Money, MTN MoMo, Wave, carte bancaire, paiement à la livraison (Abidjan uniquement), paiement en 3 fois dès 150 000 FCFA. Service client : du lundi au samedi, de 8 h à 18 h, au +225 07 00 00 00 00 (appel ou WhatsApp).


### En Python pur : décrire l'outil au LLM

Le LLM ne lit pas ton code Python. Pour qu'il sache qu'un outil existe, à quoi il sert et comment l'appeler, il faut lui envoyer une **fiche descriptive** au format JSON : le nom, l'usage et les paramètres.

In [16]:
schema_stock = {
    "type": "function",
    "function": {
        "name": "verifier_stock",
        "description": "Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures.",
        "parameters": {
            "type": "object",
            "properties": {
                "produit": {"type": "string", "description": "Nom du produit, ex. iphone 16"},
                "couleur": {"type": "string", "description": "Couleur demandée, ex. noir"},
                "taille":  {"type": "string", "description": "Taille, ex. 42 ou M (vide si sans taille)"},
            },
            "required": ["produit", "couleur"],          # la taille est facultative
        },
    },
}

Écrire cinq fiches à la main serait long et source d'erreurs de recopie. La fonction `decrire()` les **fabrique automatiquement** en lisant la docstring et la signature de chaque fonction : nom des paramètres, types, caractère obligatoire ou non.

In [17]:
import inspect
TYPES_JSON = {str: "string", float: "number", int: "integer", bool: "boolean"}

def decrire(fonction):
    """Fabrique la fiche JSON d'un outil à partir de sa docstring et de sa signature."""
    params = inspect.signature(fonction).parameters
    proprietes = {nom: {"type": TYPES_JSON[p.annotation]} for nom, p in params.items()}
    obligatoires = [nom for nom, p in params.items() if p.default is inspect.Parameter.empty]
    return {"type": "function", "function": {"name": fonction.__name__, "description": fonction.__doc__,
            "parameters": {"type": "object", "properties": proprietes, "required": obligatoires}}}

FONCTIONS = [rechercher_produits, fiche_produit, verifier_stock, delai_livraison, infos_pratiques]
SCHEMAS = [decrire(f) for f in FONCTIONS]
OUTILS = {f.__name__: f for f in FONCTIONS}            # nom -> fonction, pour les exécuter
print(json.dumps(SCHEMAS[3], ensure_ascii=False, indent=1))

{
 "type": "function",
 "function": {
  "name": "delai_livraison",
  "description": "Estime les dates de livraison standard et express à domicile, à partir de la commune d'Abidjan du client (ex. Cocody) ou de sa ville (Bouaké, Yamoussoukro, San Pedro).",
  "parameters": {
   "type": "object",
   "properties": {
    "ville": {
     "type": "string"
    }
   },
   "required": [
    "ville"
   ]
  }
 }
}


`decrire()` est un outil **pédagogique** : il ne gère que des paramètres simples, sans exemples ni listes de valeurs autorisées. En production, on écrit un schéma explicite, le plus souvent avec **Pydantic** (étape 6).

### Avec LangChain

Le décorateur `tool()` fait ce que fait `decrire()` : il lit le **nom, la docstring et les annotations de type** de la fonction pour produire la fiche, dans un format que tous les fournisseurs de LLM compris par LangChain acceptent. La documentation de référence est [ici](https://python.langchain.com/docs/concepts/tools/).

In [18]:
from langchain_core.tools import tool

outils_lc = [tool(f) for f in FONCTIONS]

for o in outils_lc:
    print(f"{o.name:<20} │ paramètres : {list(o.args)}")

rechercher_produits  │ paramètres : ['recherche', 'budget_max']
fiche_produit        │ paramètres : ['produit']
verifier_stock       │ paramètres : ['produit', 'couleur', 'taille']
delai_livraison      │ paramètres : ['ville']
infos_pratiques      │ paramètres : []


**À retenir**

Un outil, c'est **une fonction Python qui renvoie du texte**, accompagnée d'**une description** destinée au LLM. En Python pur, on écrit cette description en JSON (ou on la génère) ; LangChain la déduit de la docstring. Une bonne docstring fait un bon outil, car c'est elle qui guide le choix du modèle.

---
## Étape 4 · Le tool calling

Les outils existent et le modèle en connaît les fiches. Reste à comprendre comment il s'en sert. Le principe central, souvent mal compris, est que **le LLM n'exécute jamais rien lui-même**. Face à la question d'un client, il ne renvoie pas une réponse mais une **demande** : « appelle `verifier_stock` avec `produit="air force"`, `couleur="blanc"`, `taille="42"` ». C'est **ton code** qui exécute la fonction, puis qui renvoie le résultat au modèle. Ce protocole est le **tool calling** (ou *function calling*).

<p align="center"><img src="assets/diagrams/nb2_tool_calling.png" alt="Séquence du tool calling : le code envoie la question, le LLM demande un outil, le code l'exécute et renvoie le résultat" width="760"></p>

(1) Ton code envoie au LLM la question du client et les fiches des outils. (2) Le LLM répond par une demande d'appel, avec le nom de l'outil et ses arguments. (3) Ton code exécute la fonction. (4) Il renvoie le résultat au LLM dans un message de rôle `tool`. (5) Le LLM rédige enfin la réponse au client. Le modèle propose, le code dispose : c'est ce partage des rôles qui permet, plus loin, de contrôler ce qu'Awa peut faire.

On commence par le **prompt système** d'Awa, sa « fiche de poste ». Il lui dit quel outil utiliser pour quelle question et lui interdit d'inventer, ce qui aide notamment les petits modèles locaux.

In [19]:
SYSTEME = f"""Tu es Awa, conseillère de vente de la boutique en ligne {BOUTIQUE['nom']} ({BOUTIQUE['slogan']}).
Nous sommes le {JOURS[AUJOURDHUI.weekday()]} {AUJOURDHUI:%d/%m/%Y}. Les prix sont en FCFA.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (commune, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux."""
print(SYSTEME)

Tu es Awa, conseillère de vente de la boutique en ligne Djassa (Les bons plans du quotidien, livrés partout en Côte d'Ivoire).
Nous sommes le mardi 27/10/2026. Les prix sont en FCFA.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (commune, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux.


### 4a · Le LLM demande un outil

Premier appel : on envoie la question et les fiches des outils, et on examine ce que le modèle renvoie.

In [20]:
messages = [{"role": "system", "content": SYSTEME},
            {"role": "user", "content": "Vous avez les Nike Air Force 1 blanches en 42 ?"}]

reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
demande = reponse.choices[0].message

print("Texte du LLM :", repr(demande.content))          # souvent vide : il préfère demander un outil
for appel in demande.tool_calls or []:
    afficher("outil", f"{appel.function.name}({appel.function.arguments})")

Texte du LLM : None
Appelle      │ verifier_stock({"produit":"Nike Air Force 1","couleur":"Blanc","taille":"42"})


Le LLM n'a **pas répondu au client** : son texte est vide (`None`), et il a renvoyé une **demande d'appel** avec le nom de l'outil et ses arguments en JSON. Remarque que le modèle a déduit seul les arguments à partir de la phrase du client (« Air Force 1 » pour le produit, « blanc » pour la couleur, « 42 » pour la taille).

### 4b · Ton code exécute l'outil et renvoie le résultat

On exécute la fonction demandée, on ajoute son résultat à la conversation dans un message de rôle `tool`, puis on rappelle le modèle pour qu'il rédige la réponse.

In [21]:
def executer(nom, arguments):
    """Exécute l'outil demandé par le LLM (on la renforcera à l'étape 6)."""
    return OUTILS[nom](**arguments)

appel = demande.tool_calls[0]
arguments = json.loads(appel.function.arguments)        # texte JSON -> dictionnaire Python
resultat = executer(appel.function.name, arguments)
afficher("resultat", resultat)

messages.append(demande)                                # on garde la demande du LLM...
messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})  # ...et le résultat

final = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
afficher("reponse", final.choices[0].message.content)

Reçoit       │ baskets nike air force 1 blanc taille 42 : 8 en stock.

Répond       │ Bonne nouvelle — il y a 8 paires de Nike Air Force 1 blanches en 42 en stock chez Djassa. Voulez-vous que je vérifie le délai de livraison pour votre commune ?


### Avec LangChain

`bind_tools()` « attache » les fiches d'outils au modèle. LangChain renvoie les arguments déjà convertis en dictionnaire Python, sans avoir à relire du JSON.

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODELE, base_url=BASE_URL, api_key=CLE, **REGLAGES)   # marche aussi avec Ollama
llm_avec_outils = llm.bind_tools(outils_lc)

demande_lc = llm_avec_outils.invoke([("system", SYSTEME), ("user", "Vous avez les Nike Air Force 1 blanches en 42 ?")])
for appel in demande_lc.tool_calls:
    afficher("outil", f"{appel['name']}({appel['args']})")

Appelle      │ verifier_stock({'produit': 'Nike Air Force 1', 'couleur': 'blanc', 'taille': '42'})


**À retenir**

Le LLM **choisit** l'outil et ses arguments ; **ton code** l'exécute, ce qui te laisse le contrôle. Le résultat revient dans un message de rôle `tool`, puis le modèle rédige la réponse. Pour un nouveau projet, OpenAI recommande son API *Responses* ; on conserve ici *Chat Completions*, dont la boucle est plus lisible pour apprendre.

---
## Étape 5 · La boucle ReAct : de l'outil à l'agent

À l'étape 4, on a fait **un seul aller-retour**, orchestré à la main. Or une vraie demande en exige souvent plusieurs : pour conseiller un cadeau, Awa doit d'abord **chercher** des produits, puis **lire les avis**, puis répondre. La méthode **ReAct** (*Reason + Act*) formalise cet enchaînement : le modèle **décide** d'une action, l'application l'exécute, le modèle **observe** le résultat, et le cycle recommence jusqu'à ce que le modèle n'ait plus besoin d'outil.

<p align="center"><img src="assets/diagrams/nb2_boucle_react.png" alt="Boucle ReAct : réfléchir, agir, observer, puis répondre quand le LLM n'a plus besoin d'outil" width="860"></p>

Le cycle comporte trois temps : le LLM **réfléchit** (il choisit la suite), il **agit** (il demande un outil), il **observe** (il lit le résultat). La flèche de retour montre que l'observation nourrit la réflexion suivante. La sortie se fait quand le LLM décide de **répondre** au lieu de demander un outil. C'est cette décision d'arrêt, prise par le modèle et non par le code, qui distingue un **agent** d'un simple script enchaînant des étapes fixes.

**Ce qu'on voit réellement.** La « réflexion » se passe à l'intérieur du modèle et n'est pas observable. Ce qu'on peut observer et vérifier, ce sont ses **décisions** : quel outil, avec quels arguments, pour quel résultat. Si le modèle écrit un petit texte en plus, on l'affiche avec l'étiquette *Note*, sans le prendre pour son raisonnement véritable.

### En Python pur : une boucle `for`

La boucle ci-dessous tient en quelques lignes. À chaque tour, elle envoie la liste de messages au modèle ; si celui-ci demande des outils, elle les exécute et ajoute leurs résultats ; sinon elle s'arrête. Un garde-fou (`max_tours`) empêche une boucle sans fin.

In [23]:
def roles(messages):
    """['system', 'user', 'assistant (demande d'outil)', 'tool'] : les rôles des messages, dans l'ordre."""
    noms = [m["role"] if isinstance(m, dict) else m.role for m in messages]
    return [n + " (demande d'outil)" if not isinstance(m, dict) and m.tool_calls else n for n, m in zip(noms, messages)]

def agent_python(question, max_tours=6, montrer_etat=False):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    for tour in range(1, max_tours + 1):             # garde-fou : jamais de boucle infinie
        if montrer_etat:
            print(f"   Tour {tour} : le LLM reçoit {len(messages)} messages → {', '.join(roles(messages))}")
        reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
        msg = reponse.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:                       # plus d'outil demandé -> réponse finale
            return afficher("reponse", msg.content)
        if msg.content:
            afficher("-", msg.content)          # texte éventuel (pas son raisonnement interne)
        for appel in msg.tool_calls:
            arguments = json.loads(appel.function.arguments or "{}")
            afficher("outil", f"{appel.function.name}({arguments})")        # AGIR
            resultat = executer(appel.function.name, arguments)
            afficher("resultat", resultat)                                  # OBSERVER
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})
    afficher("reponse", "Désolée, je n'ai pas réussi à répondre. Un conseiller va vous recontacter.")

In [24]:
agent_python("Je cherche un cadeau à moins de 40 000 FCFA. Qu'est-ce qui est le mieux noté chez vous ?")

Appelle      │ rechercher_produits({'recherche': '', 'budget_max': 40000})
Reçoit       │ - sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)
Appelle      │ fiche_produit({'produit': 'sac à dos eastpak (Eastpak)'})
Reçoit       │ Produit introuvable ou ambigu : demande au client de préciser. Produits vendus : iphone 16, samsung galaxy a36, écouteurs airpods 4, montre connectée xiaomi, baskets nike air force 1, hoodie adidas essentials, jean levis 501, sac à dos eastpak, gobelet stanley quencher.

Répond       │ Bonne idée ! Le mieux noté sous 40 000 FCFA est le sac à dos Eastpak — 35 000 FCFA, note 4,7/5 (346 avis).  
Voulez-vous que je vérifie sa disponibilité par couleur/taille ou que je vous propose d'autres

### Ce que le LLM reçoit à chaque tour

La boucle ne fait finalement qu'une chose : **agrandir la liste `messages`** et la renvoyer **en entier** au modèle, exactement comme l'historique du notebook 1. À chaque tour, le LLM ne se souvient de rien et relit tout. Avec `montrer_etat=True`, on voit cette liste grandir :

In [25]:
agent_python("L'iPhone 16 noir est dispo ? Et je le reçois quand à Cocody ?", montrer_etat=True)

   Tour 1 : le LLM reçoit 2 messages → system, user
Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'Noir'})
Reçoit       │ iphone 16 noir : 8 en stock.
Appelle      │ delai_livraison({'ville': 'Cocody'})
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
   Tour 2 : le LLM reçoit 5 messages → system, user, assistant (demande d'outil), tool, tool

Répond       │ Bonjour, c’est Awa de Djassa : il y a 8 iPhone 16 noirs en stock.  
Livraison à Cocody — standard le jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA), express le mercredi 28/10 (2 500 FCFA).  
Voulez-vous que je consulte la fiche produit pour le prix, la capacité disponible ou les avis ?


La trace se lit comme un état qui évolue :

- Au **tour 1**, le LLM reçoit **2 messages** (`system`, `user`) et répond par une seule **demande d'outils**, ici deux outils à la fois (le stock et la livraison).
- Ton code exécute ces outils et **ajoute** leurs résultats, un message `tool` par résultat : la liste passe à **5 messages**.
- Au **tour 2**, le LLM relit ces 5 messages ; n'ayant plus besoin d'outil, il **répond au client**, et la boucle s'arrête parce qu'il ne demande plus rien.

Tout ce que le modèle « sait » à un instant donné se trouve dans cette liste, que **ton code** construit : la mémoire d'un agent, c'est ce qu'on lui renvoie.

### Avec LangGraph

[LangGraph](https://langchain-ai.github.io/langgraph/) construit les agents sous forme de **graphe** : notre boucle devient deux **nœuds** (des étapes de calcul) reliés par des **arêtes** (des transitions), dont l'une est conditionnelle.

<p align="center"><img src="assets/diagrams/nb2_graphe_langgraph.png" alt="Graphe LangGraph : nœud llm, condition tools_condition, nœud outils, retour au llm, fin" width="880"></p>

Le graphe démarre par le nœud `llm`, qui appelle le modèle. À la sortie, une **condition** (`tools_condition`) examine la réponse : si le modèle a demandé des outils, on va au nœud `outils`, qui les exécute puis **revient** au nœud `llm` ; sinon, le graphe se termine. On retrouve exactement la boucle écrite à la main, avec l'avantage de la voir : on pourra y insérer d'autres nœuds, par exemple une validation humaine avant un remboursement (notebook 4).

| Élément LangGraph | Rôle | Équivalent dans `agent_python` |
|---|---|---|
| `MessagesState` | L'état partagé : la liste des messages | la liste `messages` |
| nœud `llm` | Appelle le LLM avec les outils | `client.chat.completions.create(...)` |
| nœud `outils` (`ToolNode`) | Exécute les outils demandés | la boucle `for appel in ...` |
| `tools_condition` | Décide : vers les outils, ou fin | `if not msg.tool_calls` |

In [26]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def construire_agent(outils):
    llm_outils = llm.bind_tools(outils)
    def noeud_llm(etat):                              # le LLM lit l'historique et décide
        return {"messages": [llm_outils.invoke([("system", SYSTEME)] + etat["messages"])]}
    graphe = StateGraph(MessagesState)
    graphe.add_node("llm", noeud_llm)
    graphe.add_node("outils", ToolNode(outils))       # exécute les outils demandés
    graphe.add_edge(START, "llm")
    graphe.add_conditional_edges("llm", tools_condition, {"tools": "outils", END: END})
    graphe.add_edge("outils", "llm")                  # après les outils, retour au LLM
    return graphe.compile()

agent_lg = construire_agent(outils_lc)
print("Agent LangGraph prêt :", list(agent_lg.get_graph().nodes))

Agent LangGraph prêt : ['__start__', 'llm', 'outils', '__end__']


Comme pour la version Python, une petite fonction traduit l'historique des messages LangGraph en traces lisibles (Note, Appelle, Reçoit, Répond).

In [27]:
def afficher_messages(messages):
    """Traduit l'historique LangGraph en traces Note, Appelle, Reçoit et Répond."""
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            if m.content:
                afficher("pensee", m.content)
            for appel in m.tool_calls:
                afficher("outil", f"{appel['name']}({appel['args']})")
        elif m.type == "tool":
            afficher("resultat", m.content)
        elif m.type == "ai":
            afficher("reponse", m.content)

In [28]:
resultat = agent_lg.invoke({"messages": [("user", "Je cherche un cadeau à moins de 40 000 FCFA. Qu'est-ce qui est le mieux noté chez vous ?")]})
afficher_messages(resultat["messages"])

Appelle      │ rechercher_produits({'recherche': '', 'budget_max': 40000})
Reçoit       │ - sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)

Répond       │ Le mieux noté dans votre budget (≤40 000 FCFA) est le sac à dos Eastpak — 35 000 FCFA, note 4,7/5 (346 avis).  
Autres bonnes options : gobelet Stanley Quencher 28 000 FCFA (4,5/5), hoodie Adidas 38 000 FCFA (4,4/5) et montre connectée Xiaomi 32 000 FCFA (4,2/5).  
Souhaitez-vous que je vérifie la disponibilité ou vous montre la fiche détaillée d’un de ces articles ?


**À retenir**

Un **agent**, c'est un LLM, des outils et **une boucle** dont le modèle décide quand sortir. Il faut toujours un **garde-fou** contre les boucles infinies : `max_tours` en Python pur, `recursion_limit` dans LangGraph. LangGraph rend la boucle **visible** et extensible. Pour un cas standard, `create_agent` (package `langchain`) construit ce même graphe en une ligne ; on a écrit les pièces à la main pour voir ce qu'il y a dedans.

---
## Étape 6 · Sécuriser les outils avec Pydantic

L'agent fonctionne, mais il fait confiance au modèle pour les arguments. Or le LLM n'est pas infaillible : il peut envoyer `ville="Abidjan"` au lieu d'une commune comme `"Cocody"`, ou `taille=42` (un nombre) au lieu de `"42"` (un texte). Il faut donc vérifier les arguments **entre la demande du modèle et l'exécution** : c'est la place de ton programme, qui a le dernier mot.

<p align="center"><img src="assets/diagrams/nb2_validation_pydantic.png" alt="Le programme valide le format puis les règles métier avant d'exécuter l'outil, sinon l'erreur revient au LLM" width="900"></p>

La demande du LLM traverse deux contrôles avant d'atteindre l'outil. Le premier vérifie le **format** (le bon type, un champ obligatoire présent), le second les **règles métier** (la commune est-elle desservie ?). Si un contrôle échoue, on n'exécute rien et l'**erreur est renvoyée au LLM**, qui peut se corriger ou interroger le client. Si les deux passent, l'outil s'exécute. Voyons d'abord ce qui se passe sans ces contrôles :

In [29]:
print(delai_livraison("Korhogo"))                    # une ville qu'on ne dessert pas : pas d'erreur... mais une date inventée
try:
    verifier_stock("air force", "blanc", 42)         # un nombre au lieu d'un texte
except Exception as erreur:
    print("Plantage :", repr(erreur))

Standard : lundi 02/11 (offerte dès 50 000 FCFA, sinon 3 000 FCFA). Express : jeudi 29/10 (5 000 FCFA).
Plantage : AttributeError("'int' object has no attribute 'upper'")


Deux défauts apparaissent : une **réponse fausse en silence** (une date inventée pour une ville non desservie) et un **plantage** qui interromprait toute la boucle de l'agent.

[**Pydantic**](https://docs.pydantic.dev) est la bibliothèque Python de référence pour **valider des données**. On décrit la forme attendue des arguments dans une classe, et Pydantic :

- **corrige** ce qui peut l'être (`42` devient `"42"`) ;
- **refuse** ce qui est invalide, avec un message clair (« ville non desservie »).

In [30]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

VILLES = {cle(v): v for zone in ZONES.values() for v in zone["villes"]}       # les communes et villes desservies

class ArgsLivraison(BaseModel):
    ville: str = Field(description="Commune d'Abidjan (ex. Cocody, Yopougon) ou ville desservie (Bouaké, Yamoussoukro, San Pedro)")

    @field_validator("ville")
    @classmethod
    def ville_desservie(cls, valeur):
        if cle(valeur) not in VILLES:                                           # « Abidjan » seul ne suffit pas : il faut la commune
            raise ValueError("ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro")
        return VILLES[cle(valeur)]

class ArgsStock(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)       # 42 -> "42"
    produit: str = Field(description="Nom du produit, ex. iphone 16")
    couleur: str = Field(description="Couleur demandée, ex. noir")
    taille: str = Field("", description="Taille, ex. 42 ou M (vide si sans taille)")

print("Valide :", ArgsStock(produit="air force", couleur="blanc", taille=42))
try:
    ArgsLivraison(ville="Abidjan")
except ValidationError as erreur:
    print("Refusé :", erreur.errors()[0]["msg"])

Valide : produit='air force' couleur='blanc' taille='42'
Refusé : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro


### En Python pur : brancher la validation dans l'agent

On remplace `executer()` par une version qui **valide d'abord**. L'astuce est de ne pas planter en cas d'erreur, mais de **renvoyer l'erreur au LLM** comme résultat de l'outil : grâce à la boucle ReAct, il peut alors **se corriger** ou **demander l'information au client**. Comme `agent_python()` appelle déjà `executer()`, il profite tout de suite de cette version.

In [31]:
VALIDATEURS = {"delai_livraison": ArgsLivraison, "verifier_stock": ArgsStock}

def executer(nom, arguments):
    """Valide les arguments avec Pydantic, puis exécute l'outil. En cas d'erreur, prévient le LLM."""
    if nom not in OUTILS:
        return f"Outil inconnu : {nom}."
    if nom in VALIDATEURS:
        try:
            arguments = VALIDATEURS[nom](**arguments).model_dump()
        except ValidationError as erreur:
            detail = erreur.errors()[0]
            return f"Arguments invalides ({detail['loc'][0]} : {detail['msg']}). Corrige-les ou demande l'information au client."
    try:
        return OUTILS[nom](**arguments)
    except Exception as erreur:                         # panne, délai dépassé...
        return f"Erreur technique ({erreur}). N'invente rien : propose au client de réessayer plus tard."

print(executer("delai_livraison", {"ville": "Abidjan"}))
print(executer("verifier_stock", {"produit": "air force", "couleur": "blanc", "taille": 42}))

Arguments invalides (ville : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro). Corrige-les ou demande l'information au client.
baskets nike air force 1 blanc taille 42 : 8 en stock.


**À toi de prédire.** Dans la cellule suivante, le client dit seulement « J'habite à Abidjan », sans préciser sa commune. Avant d'exécuter, décide : que va répondre Awa ?

- **(a)** une date de livraison, car elle aura choisi une commune toute seule ;
- **(b)** une demande au client de préciser sa commune.

<details>
<summary>Voir la réponse</summary>

Normalement **(b)** : Pydantic refuse `Abidjan`, l'erreur revient à Awa comme résultat de l'outil, et la consigne « ne la devine jamais » lui dit de redemander. Si tu observes **(a)**, c'est que le LLM a deviné une commune (par exemple Cocody) : c'est exactement le risque décrit plus bas, et la raison de cette consigne dans le prompt.

</details>

In [32]:
agent_python("Je voudrais le sac Eastpak noir, je le reçois quand ? J'habite à Abidjan.")

Appelle      │ verifier_stock({'produit': 'Sac Eastpak', 'couleur': 'noir'})
Reçoit       │ sac à dos eastpak noir : 10 en stock.
Appelle      │ delai_livraison({'ville': 'Abidjan'})
Reçoit       │ Arguments invalides (ville : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro). Corrige-les ou demande l'information au client.

Répond       │ Le sac Eastpak noir est disponible : 10 exemplaires en stock.  
Pour estimer la date de livraison à Abidjan, pouvez-vous préciser votre commune (ex. Cocody, Yopougon, Treichville) ?  
Dès que vous me donnez la commune je vous indique les délais standard et express.  
Je suis là pour vous aider !


Le client n'a pas précisé sa **commune** d'Abidjan. Pydantic refuse l'argument `Abidjan` (ce n'est pas une commune livrable) et Awa reçoit l'erreur comme résultat de l'outil. Plutôt que d'inventer une date, elle **demande la commune** au client.

### Avec LangChain

On passe la classe Pydantic à `tool()` via `args_schema`. LangChain s'en sert à la fois pour **fabriquer la fiche JSON** (avec les descriptions des champs) et pour **valider les arguments**. Dans LangGraph, `ToolNode` renvoie automatiquement l'erreur au LLM, comme le fait notre `executer()`.

In [33]:
outils_valides = [tool(rechercher_produits), tool(fiche_produit), tool(infos_pratiques),
                  tool(verifier_stock, args_schema=ArgsStock),
                  tool(delai_livraison, args_schema=ArgsLivraison)]

agent_lg = construire_agent(outils_valides)                      # même graphe, outils sécurisés
afficher_messages(agent_lg.invoke({"messages": [("user", "Je voudrais le sac Eastpak noir, je le reçois quand ? J'habite à Abidjan.")]})["messages"])

Appelle      │ verifier_stock({'produit': 'sac Eastpak', 'couleur': 'noir', 'taille': ''})
Reçoit       │ sac à dos eastpak noir : 10 en stock.
Appelle      │ delai_livraison({'ville': 'Abidjan'})
Reçoit       │ Error invoking tool 'delai_livraison' with kwargs {'ville': 'Abidjan'} with error:
 ville: Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro
 Please fix the error and try again.

Répond       │ Bonne nouvelle — le sac Eastpak noir est disponible (10 en stock). Pour vous donner le délai de livraison à Abidjan, pouvez-vous préciser votre commune (ex. Cocody, Yopougon, Treichville) ? Dès que vous me donnez la commune, je vous indique les délais standard et express.


### Quand c'est le système qui casse

Un agent rencontre aussi des erreurs qui ne viennent pas des arguments. Chacune appelle une réponse prévue à l'avance :

| Problème | Ce que fait le programme |
|---|---|
| Outil inconnu | répond « outil inconnu » au LLM |
| Arguments JSON illisibles | renvoie l'erreur au LLM, ou s'arrête proprement |
| Catalogue indisponible, délai dépassé | renvoie « erreur technique », sans planter |
| Résultat vide ou incohérent | contrôle métier avant de répondre |

Simulons une **panne du catalogue** pour voir Awa réagir :

In [34]:
def stock_en_panne(**arguments):
    raise TimeoutError("le catalogue ne répond pas")

OUTILS["verifier_stock"] = stock_en_panne        # on simule la panne
agent_python("Vous avez l'iPhone 16 en noir ?")
OUTILS["verifier_stock"] = verifier_stock        # on répare

Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'noir'})
Reçoit       │ Erreur technique (le catalogue ne répond pas). N'invente rien : propose au client de réessayer plus tard.

Répond       │ Désolée, le catalogue rencontre actuellement une erreur technique et je ne peux pas vérifier la disponibilité de l'iPhone 16 en noir pour le moment. Pouvez-vous réessayer dans quelques minutes ? Si vous préférez, donnez une autre couleur ou modèle et je tenterai de vérifier dès que le catalogue répond. Merci beaucoup pour votre patience.


Awa n'invente pas de stock et ne réessaie pas indéfiniment (`max_tours` l'en empêche). Une boucle ReAct **ne remplace pas** les garde-fous d'un logiciel : elle s'y ajoute.

**À retenir**

Les arguments produits par un LLM sont des **données non fiables**, comme un formulaire rempli par un inconnu : on les valide toujours. En cas d'erreur, on **renvoie l'erreur au LLM** plutôt que de planter, car la boucle ReAct lui permet de se corriger. Attention cependant : « se corriger » ne doit pas vouloir dire **deviner**. Sans consigne, un modèle peut choisir une commune à la place du client (« Cocody » au lieu d'« Abidjan ») ; c'est pourquoi le prompt système dit *« demande-la : ne la devine jamais »*. Enfin, la classe Pydantic sert **à la fois** à valider et à décrire l'outil.

Pydantic ne protège toutefois qu'**un** des trois niveaux de contrôle :

| Niveau | Exemple | Protection |
|---|---|---|
| Format | `Abidjan` au lieu d'une commune desservie | Pydantic |
| Métier | taille 46 inexistante pour ce produit | l'outil, les règles métier |
| Autorisation | réserver ou rembourser | règles de l'application + validation humaine |

La conversion `42 → "42"` est pratique, mais elle ne dit pas si la taille existe, ni si la demande est légitime.

---
## Étape 7 · Le scénario complet

On met maintenant l'agent à l'épreuve avec un vrai message client qui contient **plusieurs questions en une**. Awa doit décider seule quels outils utiliser, et le scénario est piégé : l'iPhone rose est en rupture, et la livraison standard vers Cocody n'arrive pas avant jeudi. Saura-t-elle proposer une autre couleur et l'option express ?

In [35]:
CLIENT = ("Bonjour ! L'iPhone 16 rose est dispo ? Sinon, quelles couleurs vous reste-t-il ? "
          "J'habite à Cocody : je peux l'avoir avant jeudi ? Et je peux le renvoyer s'il ne me plaît pas ?")

print("════ Version Python pur ════")
agent_python(CLIENT)

════ Version Python pur ════
Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose'})
Reçoit       │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
Appelle      │ fiche_produit({'produit': 'iPhone 16'})
Reçoit       │ IPHONE 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Fatou, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Kouamé, 4/5)
Appelle      │ delai_livraison({'ville': 'Cocody'})
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Appelle      │ infos_pratiques({})
Reçoit       │ Retours : Retour sous 14 jours, article non u

In [36]:
print("════ Version LangGraph ════")
afficher_messages(agent_lg.invoke({"messages": [("user", CLIENT)]})["messages"])

════ Version LangGraph ════
Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose', 'taille': ''})
Appelle      │ fiche_produit({'produit': 'iPhone 16'})
Appelle      │ delai_livraison({'ville': 'Cocody'})
Appelle      │ infos_pratiques({})
Reçoit       │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
Reçoit       │ IPHONE 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Fatou, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Kouamé, 4/5)
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Reçoit       │ Retours : Retour sous 14 jours, 

Dans la trace LangGraph, plusieurs lignes *Appelle* arrivent **d'un coup** avant les lignes *Reçoit* : le LLM peut demander plusieurs outils **dans le même tour** (stock, livraison, retours). Le programme les exécute tous, puis lui renvoie tous les résultats.

<p align="center"><img src="assets/diagrams/nb2_scenario_complet.png" alt="Un message client de quatre questions déclenche quatre appels d'outils dans le même tour, puis une seule réponse" width="860"></p>

Le message du client, à gauche, contient quatre questions. Awa les distribue à quatre outils **en parallèle**, dans un même tour, au lieu d'attendre un résultat pour demander le suivant. Les quatre résultats reviennent ensemble, et Awa rédige **une seule réponse** qui les combine. Cette parallélisation réduit le nombre d'appels au modèle, donc le coût et la latence.

**Python pur et LangGraph ne donnent pas toujours la même réponse**, car le LLM n'est pas déterministe. Si tu essaies `FOURNISSEUR = "ollama"` à l'étape 0, tu verras qu'un petit modèle local comme qwen2.5:7b **oublie parfois un outil**, par exemple la livraison : il répond alors sans avoir vérifié. La leçon est générale : **la fiabilité d'un agent dépend du modèle**, mais aussi de la clarté des descriptions d'outils, du prompt, des données et du traitement des erreurs.

### Teste tes propres questions

Modifie la phrase de la cellule suivante et relance-la. Quelques idées :

- *« Le hoodie Adidas beige en M, vous l'avez ? »* : la taille est en rupture, Awa propose-t-elle une autre taille ?
- *« Les AirPods 4 valent le coup ? Que disent les clients ? »*
- *« Qu'est-ce que vous avez en promo en ce moment ? »*
- *« Vous vendez des PlayStation 5 ? »* : le produit n'est pas au catalogue, Awa invente-t-elle ?
- *« Vous avez les Nike Air Max en 42 ? »* : Awa les confond-elle avec les Air Force 1 ?

In [37]:
ma_question = "Le hoodie Adidas beige en M, vous l'avez ?"
afficher_messages(agent_lg.invoke({"messages": [("user", ma_question)]})["messages"])

Appelle      │ verifier_stock({'produit': 'hoodie Adidas', 'couleur': 'beige', 'taille': 'M'})
Reçoit       │ hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.

Répond       │ Bonjour — c’est Awa. Le hoodie Adidas beige n'est pas disponible en taille M ; les tailles en stock sont S, L et XL. Voulez-vous que je cherche le même modèle dans une autre couleur, une autre taille, ou des hoodies Adidas similaires ?


---
## Synthèse

**Les quatre idées à garder.** Le LLM ne connaît pas la boutique : **les outils lui donnent accès aux vraies données**. Il **ne fait que demander** les appels, et c'est ton code qui les exécute, ce qui préserve ton contrôle. Ce qu'il envoie **se valide** (Pydantic) avant exécution, et les actions à risque exigent d'autres garde-fous. Enfin, les frameworks **n'ont rien de magique** : ils automatisent la boucle que tu as écrite à la main.

**Repères**

| Concept | Python pur | Framework |
|---|---|---|
| Outil | `def` + fiche JSON (`decrire`) | `tool(fonction)` (LangChain) |
| Tool calling | `tools=SCHEMAS` | `llm.bind_tools()` (LangChain) |
| Boucle ReAct | boucle `for` | `StateGraph` + `ToolNode` (LangGraph) |
| Validation | `executer()` + Pydantic | `tool(..., args_schema=...)` |
| Agent | `agent_python()` | `agent_lg` |

**Prochain notebook.** Les outils de ce notebook interrogent des **données structurées** (un stock, un prix). Mais beaucoup de questions portent sur des **textes** : conditions de retour, garantie, modes de paiement, guide des tailles. Le notebook 3 apprend à Awa à lire de la documentation et à y retrouver le bon passage, c'est le **RAG**.